# Chowdeck — Tính toán KPI

Tính các KPI chính thức (Revenue, AOV, tương quan...) và xuất bảng tổng hợp. Cần chạy 01a_chowdeck_cleaning trước.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 50)

## ĐƯỜNG DẪN

In [2]:
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    BASE_DIR = Path.cwd().resolve().parent
CLEAN_PATH = BASE_DIR / 'data' / 'clean' / 'chowdeck_clean.csv'
KPI_DIR = BASE_DIR / 'outputs' / 'kpi_summary'
KPI_DIR.mkdir(parents=True, exist_ok=True)

## 1. ĐỌC DỮ LIỆU ĐÃ LÀM SẠCH

In [3]:
# Cần chạy 01a_chowdeck_cleaning trước để có file này
df_clean = pd.read_csv(CLEAN_PATH, parse_dates=['Order Received', 'Order Delivered'])
print("Shape dữ liệu đã làm sạch:", df_clean.shape)

Shape dữ liệu đã làm sạch: (3061, 33)


## 2. KPI CƠ BẢN: ORDERS, REVENUE, AOV

In [4]:
total_orders = len(df_clean)
total_revenue = df_clean['revenue'].sum()
aov_restaurant = df_clean['revenue'].mean()      # AOV nhà hàng = Total Revenue / Total Orders
aov_customer = df_clean['Total'].mean()          # AOV khách hàng (gồm phí ship/dịch vụ)
avg_rating = df_clean['Rating'].mean()

print(f"Total Orders: {total_orders:,}")
print(f"Total Revenue: ₦{total_revenue:,.0f}")
print(f"AOV (nhà hàng): ₦{aov_restaurant:,.0f}")
print(f"AOV (khách hàng): ₦{aov_customer:,.0f}")
print(f"Rating trung bình: {avg_rating:.2f}")
print(f"Kiểm tra công thức Revenue = Orders × AOV: "
      f"{total_orders} × {aov_restaurant:.0f} ≈ {total_orders * aov_restaurant:,.0f} "
      f"(so với Total Revenue thực = {total_revenue:,.0f})")

Total Orders: 3,061
Total Revenue: ₦25,226,000
AOV (nhà hàng): ₦8,241
AOV (khách hàng): ₦9,413
Rating trung bình: 3.90
Kiểm tra công thức Revenue = Orders × AOV: 3061 × 8241 ≈ 25,226,000 (so với Total Revenue thực = 25,226,000)


## 3. KPI TƯƠNG QUAN (cần shop_stats để tính ở cấp nhà hàng)

In [5]:
shop_stats = df_clean.groupby('Shop Name').agg(
    total_orders=('revenue', 'count'),
    aov_restaurant=('revenue', 'mean'),
    avg_rating=('Rating', 'mean')
)
corr_rating_orders = shop_stats['avg_rating'].corr(shop_stats['total_orders'])
corr_rating_aov = shop_stats['avg_rating'].corr(shop_stats['aov_restaurant'])
corr_delivery_rating = df_clean['delivery_time_min'].corr(df_clean['Rating'])
corr_prep_rating = df_clean['prep_time_min'].corr(df_clean['Rating'])

## 4. KPI CHẤT LƯỢNG DỮ LIỆU

In [6]:
pct_time_valid = df_clean['time_sequence_valid'].mean() * 100
avg_delay = df_clean['delay_min'].mean()
pct_delayed = (df_clean['delay_min'] > 0).mean() * 100

# Với đơn có timestamp sai thứ tự, delay_min gần như luôn = 0 (giờ giao trùng giờ dự kiến) nên không phản ánh thực tế.
# Vì vậy tính thêm các KPI độ trễ CHỈ trên đơn có timestamp hợp lệ.
df_valid_ts = df_clean[df_clean['time_sequence_valid']]
avg_delay_valid = df_valid_ts['delay_min'].mean()
pct_delayed_valid = (df_valid_ts['delay_min'] > 0).mean() * 100

## 5. TỔNG HỢP VÀ LƯU KPI SUMMARY

In [7]:
kpi_summary = {
    'dataset': 'Chowdeck',
    'market': 'Nigeria',
    'total_orders': total_orders,
    'total_revenue': total_revenue,
    'aov_restaurant': aov_restaurant,
    'aov_customer': aov_customer,
    'avg_rating': avg_rating,
    'corr_rating_vs_orders(shop_level)': corr_rating_orders,
    'corr_rating_vs_aov_restaurant(shop_level)': corr_rating_aov,
    'corr_delivery_time_vs_rating': corr_delivery_rating,
    'corr_prep_time_vs_rating': corr_prep_rating,
    'pct_time_sequence_valid': pct_time_valid,
    'avg_delay_min': avg_delay,
    'pct_orders_delayed': pct_delayed,
    'avg_delay_min_valid_only': avg_delay_valid,
    'pct_orders_delayed_valid_only': pct_delayed_valid,
}
pd.DataFrame([kpi_summary]).to_csv(KPI_DIR / 'chowdeck_kpi_summary.csv', index=False)

print("\n=== HOÀN TẤT TÍNH KPI CHOWDECK ===")
print("Đã lưu outputs/kpi_summary/chowdeck_kpi_summary.csv")


=== HOÀN TẤT TÍNH KPI CHOWDECK ===
Đã lưu outputs/kpi_summary/chowdeck_kpi_summary.csv
